# Training your own Sentry detection model

This notebook fine-tunes YOLOv8 on photos from **your** camera, so it learns
your driveway, your lighting, and your neighbour's cat.

**Before you start, read this honestly:**

The stock YOLOv8 model already detects people, cars and animals well. Training
your own is worth it for two things:

1. **Packages.** The stock model has no idea what a delivered parcel is.
   This is the big win.
2. **Fewer false alarms** on the specific things that fool it at your house.

If you haven't collected images yet, this notebook has nothing to learn from.
Go and run the Outpost with `--collect` for a week first.

---

**Set the runtime to a GPU before running anything:**
`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`

It's free, and it's roughly 20x faster than the CPU option.


## 1. Check we actually got a GPU


In [ ]:
!nvidia-smi

# If this says 'command not found', you're on a CPU runtime.
# Go to Runtime -> Change runtime type and pick T4 GPU, then run again.


## 2. Install YOLOv8


In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()


## 3. Upload your dataset

You need the folder that `autolabel.py` produced (and that you then corrected
in a labelling tool). Zip it on your computer first:

```bash
zip -r dataset.zip dataset/
```

Then run the cell below and pick that zip file.


In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()          # pick dataset.zip

name = list(uploaded.keys())[0]
with zipfile.ZipFile(name) as archive:
    archive.extractall('.')

print('Extracted. Top-level contents:')
print(os.listdir('.'))


## 4. Point the trainer at the dataset

`dataset.yaml` records an absolute path from the machine that made it, which
is wrong now that we're on Colab. This rewrites it.


In [ ]:
from pathlib import Path

DATASET = Path('dataset')          # change if your folder is named differently

assert (DATASET / 'images' / 'train').is_dir(), \
    f"No images/train inside {DATASET} - check the zip's structure."

yaml_text = f'''path: {DATASET.resolve()}
train: images/train
val: images/val

names:
  0: motion
  1: person
  2: vehicle
  3: package
  4: animal
'''
(DATASET / 'dataset.yaml').write_text(yaml_text)

n_train = len(list((DATASET / 'images' / 'train').glob('*')))
n_val = len(list((DATASET / 'images' / 'val').glob('*')))
print(f'{n_train} training images, {n_val} validation images')

if n_train < 100:
    print('\nHeads up: under 100 training images usually is not enough to')
    print('beat the stock model. Collect more before trusting the result.')


## 5. Train

`yolov8n` ('n' for nano) is the smallest model. That matters here - it has to
run on modest hardware, not a gaming PC. Bigger models are more accurate but
too slow on small boards.

**Epochs** = how many times it looks at every photo. 50 is a sensible start.
More is not always better: past a point the model memorises your exact photos
instead of learning what a person looks like ("overfitting").

On a free T4 this takes roughly 10-20 minutes for a few hundred images.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')         # start from the pretrained model, don't start blank

results = model.train(
    data=str(DATASET / 'dataset.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,                   # stop early if it stops improving
    project='sentry',
    name='run',
    exist_ok=True,
)

print('Weights saved to:', results.save_dir)


## 6. Did it actually get better?

The number that matters is **mAP50** - roughly, how often it finds the right
thing in the right place. Higher is better; 1.0 is perfect.

Judge it per class. It's normal for `person` to score well and `package` to
score badly at first, simply because you have far fewer parcel photos.


In [ ]:
metrics = model.val()

print(f'Overall mAP50:    {metrics.box.map50:.3f}')
print(f'Overall mAP50-95: {metrics.box.map:.3f}')
print()

names = ['motion', 'person', 'vehicle', 'package', 'animal']
for i, class_name in enumerate(names):
    try:
        print(f'  {class_name:<8} mAP50 = {metrics.box.ap50[i]:.3f}')
    except (IndexError, TypeError):
        print(f'  {class_name:<8} no examples in the validation set')


## 7. Look at what it predicts

Numbers hide things. Always look at real predictions before trusting a model.


In [ ]:
import cv2, glob
from IPython.display import Image, display

val_images = sorted(glob.glob(str(DATASET / 'images' / 'val' / '*')))[:5]

for path in val_images:
    prediction = model(path, verbose=False)[0]

    # plot() returns the image with boxes drawn on it, as a numpy array.
    # Write it to a file so Colab can display it.
    annotated = prediction.plot()
    out_path = 'preview.jpg'
    cv2.imwrite(out_path, annotated)

    print(path.split('/')[-1])
    for box in prediction.boxes:
        print(f'   {prediction.names[int(box.cls)]}  {float(box.conf):.2f}')
    if len(prediction.boxes) == 0:
        print('   (nothing detected)')

    display(Image(filename=out_path))


## 8. Download the trained model

`best.pt` is the best-performing version, not just the last one.


In [ ]:
from google.colab import files

best = results.save_dir / 'weights' / 'best.pt'
print('Size:', round(best.stat().st_size / 1e6, 1), 'MB')

files.download(str(best))


## 9. Put it on the Outpost

Copy `best.pt` onto the Outpost, then start the agent pointing at it:

```bash
scp best.pt YOUR_USER@outpost.local:~/sentry/outpost/sentry_best.pt

# on the Outpost:
python3 outpost_agent.py --key YOUR_DEVICE_KEY \
                     --server https://your-backend-url \
                     --model sentry_best.pt
```

The agent notices the model uses Sentry's own class names and skips the
COCO translation automatically - no other changes needed.

If it turns out worse than the stock model, just drop the `--model` flag to
go back. Keep collecting images and train again with more data - that is
almost always what fixes it.
